In [1]:
# ============================================================
# NOTEBOOK 01: EXPLORATORY DATA ANALYSIS & PREPROCESSING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# ------------------------------------------------------------
# STEP 1: Load and Merge Data
# ------------------------------------------------------------
# Load raw datasets from Kaggle
train = pd.read_csv('../data/raw/train.csv')
store = pd.read_csv('../data/raw/store.csv')

# Merge sales data with store metadata on Store ID
# This adds store characteristics (type, location, competition) to each sale
df = train.merge(store, on='Store', how='left')

# Convert Date string to datetime object for time-based operations
df['Date'] = pd.to_datetime(df['Date'])

# Sort chronologically (CRITICAL for time series)
df = df.sort_values('Date')

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

C:\Users\USER\AppData\Local\Temp\ipykernel_21060\3058837211.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/raw/train.csv')


Dataset shape: (1017209, 18)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


In [3]:
# ------------------------------------------------------------
# STEP 2: Clean Data (Remove Closed Stores)
# ------------------------------------------------------------
# Stores with Open=0 have Sales=0 (closed days)
# These add noise and must be removed for accurate modeling
print(f"Rows before cleaning: {len(df)}")
df = df[df['Open'] == 1]
print(f"Rows after removing closed days: {len(df)}")

Rows before cleaning: 1017209
Rows after removing closed days: 844392


In [4]:
# ------------------------------------------------------------
# STEP 3: Handle Missing Values (SAFE METHOD)
# ------------------------------------------------------------
# CompetitionDistance: NaN means "no competitor nearby"
# Fill with median (robust to outliers) rather than 0
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(
    df['CompetitionDistance'].median()
)

# All other NaNs (promo dates, etc.) → 0 means "not applicable"
df = df.fillna(0)

# Verify no missing values remain
print("Missing values per column:")
print(df.isnull().sum().sum())  # Should be 0

Missing values per column:
0


In [5]:
# ------------------------------------------------------------
# STEP 4: Feature Engineering (MOST IMPORTANT)
# ------------------------------------------------------------
# Extract time components - sales are highly seasonal
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday

# Weekend indicator (shopping behavior changes)
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Clear promo indicator
df['IsPromo'] = df['Promo']

In [6]:
# ------------------------------------------------------------
# STEP 5: Encode Categorical Variables
# ------------------------------------------------------------
# ML models need numbers, not text categories
# One-hot encoding creates binary columns for each category
df = pd.get_dummies(
    df,
    columns=['StoreType', 'Assortment', 'StateHoliday'],
    drop_first=True  # Avoid dummy variable trap
)

# Convert boolean columns to integers (XGBoost requirement)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [7]:
# ------------------------------------------------------------
# STEP 6: Time-Based Train/Test Split (CRITICAL)
# ------------------------------------------------------------
# NEVER use random split for time series - causes data leakage
# Train on past, test on future (mimics real forecasting)
train_df = df[df['Date'] < '2015-01-01']   # 2013-2014
test_df = df[df['Date'] >= '2015-01-01']    # 2015

print(f"Training period: {train_df['Date'].min()} to {train_df['Date'].max()}")
print(f"Testing period: {test_df['Date'].min()} to {test_df['Date'].max()}")

Training period: 2013-01-01 00:00:00 to 2014-12-31 00:00:00
Testing period: 2015-01-01 00:00:00 to 2015-07-31 00:00:00


In [8]:
# ------------------------------------------------------------
# STEP 7: Define Features and Target
# ------------------------------------------------------------
# Features (X): everything except target and date
# Target (y): Sales (what we want to predict)

# CRITICAL: Remove data leakage columns
# - Customers: unknown at prediction time
# - Open: always 1 after filtering
# - PromoInterval: text, redundant with other promo features

drop_cols = ['Sales', 'Date', 'Customers', 'Open', 'PromoInterval']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['Sales']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['Sales']

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeatures: {list(X_train.columns)}")


Training set: (648360, 25)
Test set: (196032, 25)

Features: ['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'IsPromo', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'Assortment_b', 'Assortment_c', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c']


In [9]:
# ------------------------------------------------------------
# STEP 8: Save Processed Data for Modeling Notebook
# ------------------------------------------------------------
# This solves the "variable not found" error between notebooks
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("\n✅ Data saved to data/processed/")


✅ Data saved to data/processed/
